In [1]:
!pip -q install python-dotenv openai

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads .env from current directory
assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY not found in .env"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI key loaded ✅")


OpenAI key loaded ✅


In [3]:
import pandas as pd

sec_df = pd.read_json("../data/sections_extracted.jsonl", lines=True)
print(sec_df.shape)
sec_df.head()


(127, 4)


,id,section_type,heading,section_text
0,2512.19725,fallback,fallback_window,"Conclusion\nIn this work, we studied the integ..."
1,2503.17793,fallback,fallback_window,Conclusion & Future Work\nWe introduce Ling-Co...
2,2601.03085,future_work,CONCLUSIONS AND FUTURE DIRECTIONS,The growing popularity of IIoT systems present...
3,2102.03018,fallback,fallback_window,Conclusion\nThe encouragement towards training...
4,2401.17544,fallback,fallback_window,discussions. 2.1 Integer Quantization Earlier ...


In [4]:
import json

GAP_SCHEMA = {
    "type": "object",
    "properties": {
        "paper_id": {"type": "string"},
        "items": {
            "type": "array",
            "maxItems": 2,
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string", "enum": ["limitation", "future_work", "open_problem"]},
                    "gap_sentence": {"type": "string"},
                    "paragraph_text": {"type": "string"},
                    "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                },
                "required": ["type", "gap_sentence", "paragraph_text", "confidence"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["paper_id", "items"],
    "additionalProperties": False,
}

SYSTEM = "Extract up to 2 research gaps from given text. Use verbatim sentences and paragraphs from input only."

def build_user_prompt(paper_id: str, text: str) -> str:
    return (
        f"id={paper_id}\n"
        "Return JSON per schema. items<=2. "
        "type in {limitation,future_work,open_problem}. "
        "gap_sentence: one verbatim sentence stating missing/next work. "
        "paragraph_text: verbatim paragraph containing it. "
        "Skip contributions/results.\n"
        f"TEXT:\n{text}"
    )

def paper_text_from_sec_df(g: pd.DataFrame, max_chars: int = 9000) -> str:
    order = {"limitations": 0, "future_work": 1, "fallback": 2}
    gg = g.copy()
    gg["ord"] = gg["section_type"].map(lambda x: order.get(x, 9))
    gg = gg.sort_values("ord")

    chunks = []
    for _, r in gg.iterrows():
        txt = str(r["section_text"]).strip()
        if txt:
            chunks.append(txt)
    return "\n\n".join(chunks)[:max_chars]

MODEL = "gpt-4.1-mini"


In [8]:
from tqdm import tqdm

OUT_TSV = "../data/gaps_openai.tsv"  # <- final output
MAX_PAPERS = 50  # set to 10/100 for testing, or None for all

# Load existing output if resuming
if os.path.exists(OUT_TSV):
    existing = pd.read_csv(OUT_TSV, sep="\t")
    done_ids = set(existing["id"].astype(str).unique())
    print(f"Resuming: found {len(existing)} rows from {len(done_ids)} papers in {OUT_TSV}")
else:
    existing = pd.DataFrame()
    done_ids = set()

paper_ids = sec_df["id"].astype(str).unique().tolist()
if MAX_PAPERS is not None:
    paper_ids = paper_ids[:MAX_PAPERS]

paper_ids = [pid for pid in paper_ids if pid not in done_ids]
print("Papers to process:", len(paper_ids))

rows = []  # buffer, flush in chunks
FLUSH_EVERY = 25

def flush_rows(rows_buf):
    if not rows_buf:
        return
    df_new = pd.DataFrame(rows_buf)
    header = not os.path.exists(OUT_TSV)
    df_new.to_csv(OUT_TSV, sep="\t", index=False, mode="a", header=header, encoding="utf-8")
    rows_buf.clear()

for paper_id in tqdm(paper_ids):
    g = sec_df[sec_df["id"].astype(str) == paper_id]
    text = paper_text_from_sec_df(g)

    # Skip if no usable text
    if not text or len(text.strip()) < 200:
        continue

    try:
        resp = client.responses.create(
            model=MODEL,
            input=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": build_user_prompt(paper_id, text)},
            ],
            text={
                "format": {
                    "type": "json_schema",
                    "name": "gap_extraction",
                    "schema": GAP_SCHEMA,
                    "strict": True,
                }
            },
        )

        data = json.loads(resp.output_text)
        items = data.get("items", [])

        for it in items:
            rows.append({
                "id": paper_id,
                "gap_type": it["type"],
                "gap_sentence": it["gap_sentence"],
                "paragraph_text": it["paragraph_text"],
                "confidence": it["confidence"],
            })

        # flush periodically
        if len(rows) >= FLUSH_EVERY:
            flush_rows(rows)

    except Exception as e:
        # don’t crash the run
        continue

# final flush
flush_rows(rows)

print("Saved:", OUT_TSV)


Resuming: found 1 rows from 1 papers in ../data/gaps_openai.tsv
Papers to process: 49


100%|██████████| 49/49 [03:28<00:00,  4.26s/it]

Saved: ../data/gaps_openai.tsv


In [9]:
gaps = pd.read_csv(OUT_TSV, sep="\t")
print("gaps:", gaps.shape)
display(gaps.head(20))


gaps: (62, 5)


,id,gap_type,gap_sentence,paragraph_text,confidence
0,2512.19725,future_work,"In particular, our study highlights the need f...",These findings provide concrete guidance for p...,0.95
1,2503.17793,future_work,The future work will focus on further pushing ...,The future work will focus on further pushing ...,0.95
2,2503.17793,future_work,We plan to improve reasoning performance with ...,The future work will focus on further pushing ...,0.95
3,2102.03018,future_work,We plan to extend this task by performing simi...,6. Future Work and Challenges\nWe plan to exte...,0.99
4,2102.03018,limitation,Most of the challenges got raised due to many ...,We faced many challenges when ﬁnishing this ta...,0.95
5,2401.17544,future_work,Users can also define customized gradient func...,One may notice that the casting function lever...,0.95
6,2207.09511,future_work,One avenue involves exploring the connections ...,"Finally, we note some potential research direc...",0.95
7,2207.09511,future_work,Another direction involves the challenge of op...,"Finally, we note some potential research direc...",0.95
8,2508.13182,open_problem,This addresses a critical gap in research port...,"While recent advances leverage LLMs, RAG, and ...",0.95
9,2508.16261,future_work,"Therefore, applying existing efficiency-based ...",Federated DPO. DPO aligns LLMs by training on ...,0.95
